In [19]:
import pandas as pd
ts_data=pd.read_parquet("../data/transformed/ts_data_2026_07.parquet")
ts_data

,pickup_hour,rides,pickup_location_id
0,2026-07-01 00:00:00,10,4
1,2026-07-01 01:00:00,12,4
2,2026-07-01 02:00:00,7,4
3,2026-07-01 03:00:00,2,4
4,2026-07-01 04:00:00,5,4
...,...,...,...
193435,2026-07-31 19:00:00,0,84
193436,2026-07-31 20:00:00,0,84
193437,2026-07-31 21:00:00,0,84
193438,2026-07-31 22:00:00,0,84


In [20]:
# filtering out for one location (central park)
ts_data_one_location = ts_data.loc[ts_data.pickup_location_id == 43, :].reset_index(drop=True)
ts_data_one_location

,pickup_hour,rides,pickup_location_id
0,2026-07-01 00:00:00,12,43
1,2026-07-01 01:00:00,6,43
2,2026-07-01 02:00:00,2,43
3,2026-07-01 03:00:00,0,43
4,2026-07-01 04:00:00,2,43
...,...,...,...
739,2026-07-31 19:00:00,84,43
740,2026-07-31 20:00:00,111,43
741,2026-07-31 21:00:00,74,43
742,2026-07-31 22:00:00,33,43


In [21]:
# getting_cutoff_indices to be used for slicing step in slide and slice method

def get_cutoff_indices(
        data: pd.DataFrame,
        n_features: int,
        step_size: int,

) -> list:

    stop_position = len(data) -1 

    # start the first sub-sequence at position 0
    subseq_first_idx = 0
    subseq_mid_idx = n_features
    subseq_last_idx = n_features + 1
    indices = []

    while subseq_last_idx <= stop_position:

        indices.append((subseq_first_idx, subseq_mid_idx, subseq_last_idx))

        subseq_first_idx += step_size
        subseq_mid_idx += step_size
        subseq_last_idx += step_size

    return indices

In [22]:
n_features = 24 # using the whole previous 24 hours as features
step_size = 1

indices = get_cutoff_indices(
    ts_data_one_location,
    n_features,
    step_size
)
indices

indices [:5]

[(0, 24, 25), (1, 25, 26), (2, 26, 27), (3, 27, 28), (4, 28, 29)]

In [23]:
# implement the slicing 
import numpy as np
n_examples= len(indices)
x= np.ndarray(shape=(n_examples, n_features), dtype = np.float32)
y= np.ndarray(shape=(n_examples), dtype= np.float32)
pickup_hours= []

for i, idx in enumerate(indices):
    x[i,:]= ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
    y[i]= ts_data_one_location.iloc[idx[1]]['rides']
    pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])


In [24]:
print(f'{x.shape=}')
print(f'{x=}')
print(f'{y.shape=}')
print(f'{y=}')
print(f'{pickup_hours=}')



x.shape=(719, 24)
x=array([[ 12.,   6.,   2., ...,  49.,  36.,  12.],
       [  6.,   2.,   0., ...,  36.,  12.,   5.],
       [  2.,   0.,   2., ...,  12.,   5.,   3.],
       ...,
       [ 45.,  55.,  55., ..., 138., 135.,  84.],
       [ 55.,  55.,  37., ..., 135.,  84., 111.],
       [ 55.,  37.,  22., ...,  84., 111.,  74.]],
      shape=(719, 24), dtype=float32)
y.shape=(719,)
y=array([  5.,   3.,   3.,   0.,   2.,   4.,   9.,  36.,  38.,  39.,  65.,
        66.,  90., 135., 142., 144., 183., 114.,  82.,  54.,  43.,  33.,
        26.,  13.,   8.,   2.,   2.,   1.,   1.,   4.,   7.,  14.,  15.,
        15.,  30.,  60., 100., 120., 137., 114., 143., 119.,  77.,  99.,
        82.,  38.,  21.,  14.,   7.,   5.,   2.,   0.,   0.,   1.,   4.,
        14.,  10.,  23.,  27.,  40.,  56., 126., 125., 111., 131., 102.,
        92.,  59.,  54.,  32.,  16.,   8.,   1.,   5.,   3.,   2.,   1.,
         2.,   5.,   3.,  11.,  17.,  31.,  47.,  50.,  79.,  83.,  80.,
        78.,  63.,  54.,  42

In [25]:
# transform x as numpy array into pandas dataframe
features_one_location= pd.DataFrame(
    x, 
    columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(n_features))]
)
features_one_location

,rides_previous_24_hour,rides_previous_23_hour,rides_previous_22_hour,rides_previous_21_hour,rides_previous_20_hour,rides_previous_19_hour,rides_previous_18_hour,rides_previous_17_hour,rides_previous_16_hour,rides_previous_15_hour,...,rides_previous_10_hour,rides_previous_9_hour,rides_previous_8_hour,rides_previous_7_hour,rides_previous_6_hour,rides_previous_5_hour,rides_previous_4_hour,rides_previous_3_hour,rides_previous_2_hour,rides_previous_1_hour
0,12.0,6.0,2.0,0.0,2.0,3.0,16.0,24.0,43.0,46.0,...,95.0,86.0,90.0,61.0,72.0,67.0,46.0,49.0,36.0,12.0
1,6.0,2.0,0.0,2.0,3.0,16.0,24.0,43.0,46.0,49.0,...,86.0,90.0,61.0,72.0,67.0,46.0,49.0,36.0,12.0,5.0
2,2.0,0.0,2.0,3.0,16.0,24.0,43.0,46.0,49.0,74.0,...,90.0,61.0,72.0,67.0,46.0,49.0,36.0,12.0,5.0,3.0
3,0.0,2.0,3.0,16.0,24.0,43.0,46.0,49.0,74.0,75.0,...,61.0,72.0,67.0,46.0,49.0,36.0,12.0,5.0,3.0,3.0
4,2.0,3.0,16.0,24.0,43.0,46.0,49.0,74.0,75.0,86.0,...,72.0,67.0,46.0,49.0,36.0,12.0,5.0,3.0,3.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714,71.0,78.0,45.0,55.0,55.0,37.0,22.0,1.0,3.0,0.0,...,54.0,64.0,50.0,80.0,100.0,124.0,172.0,134.0,128.0,138.0
715,78.0,45.0,55.0,55.0,37.0,22.0,1.0,3.0,0.0,3.0,...,64.0,50.0,80.0,100.0,124.0,172.0,134.0,128.0,138.0,135.0
716,45.0,55.0,55.0,37.0,22.0,1.0,3.0,0.0,3.0,6.0,...,50.0,80.0,100.0,124.0,172.0,134.0,128.0,138.0,135.0,84.0
717,55.0,55.0,37.0,22.0,1.0,3.0,0.0,3.0,6.0,8.0,...,80.0,100.0,124.0,172.0,134.0,128.0,138.0,135.0,84.0,111.0


In [26]:
# transform x as numpy array into pandas dataframe
targets_one_location= pd.DataFrame(y, columns=[f'target_rides_next_hour'])
targets_one_location

,target_rides_next_hour
0,5.0
1,3.0
2,3.0
3,0.0
4,2.0
...,...
714,135.0
715,84.0
716,111.0
717,74.0


In [33]:
''' creating a function to integrate all this data processing, converting the time series data 
into tabular data for all locations '''

from tqdm import tqdm

def transform_ts_data_into_features_and_target(
        ts_data: pd.DataFrame,
        input_seq_len: int,
        step_size: int
) -> pd.DataFrame:
    """
    slices and transposes a data from time-series format into a (features, target)
    format that we can use to train supervised ML models
    """

    assert set(ts_data.columns)== {'pickup_hour', 'rides', 'pickup_location_id'}

    location_ids= ts_data['pickup_location_id'].unique()
    features= pd.DataFrame()
    targets= pd.DataFrame()

    for location_id in tqdm(location_ids):

        # keep only ts data for this 'location_id'
        ts_data_one_location= ts_data.loc[
            ts_data.pickup_location_id== location_id, ['pickup_hour', 'rides']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices= get_cutoff_indices(
            ts_data_one_location,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples= len(indices)
        n_examples= len(indices)
        x= np.ndarray(shape=(n_examples, input_seq_len), dtype = np.float32)
        y= np.ndarray(shape=(n_examples), dtype= np.float32)
        pickup_hours= []

        for i, idx in enumerate(indices):
            x[i,:]= ts_data_one_location.iloc[idx[0]:idx[1]]['rides'].values
            y[i]= ts_data_one_location.iloc[idx[1]]['rides']
            pickup_hours.append(ts_data_one_location.iloc[idx[1]]['pickup_hour'])

        # numpy -> pandas
        features_one_location= pd.DataFrame(
            x,
            columns=[f'rides_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )

        # numpy -> pandas
        targets_one_location= pd.DataFrame(y, columns=[f'target_rides_next_hour'])

        # concatenate results
        features= pd.concat([features, features_one_location])
        targets= pd.concat([targets, targets_one_location])

    features.reset_index(inplace= True, drop= True)
    targets.reset_index(inplace= True, drop= True)

    return features, targets['target_rides_next_hour']


In [34]:
features, targets= transform_ts_data_into_features_and_target(
    ts_data,
    input_seq_len= 24*7*1, # one week of history
    step_size= 24
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

100%|██████████| 260/260 [00:03<00:00, 65.71it/s]

features.shape=(6240, 168)
targets.shape=(6240,)
